# 1. Data Preparation & Exploratory Data Analysis (EDA)
This notebook covers downloading the RAVDESS dataset from HuggingFace, data ingestion, and exploratory data analysis.

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datasets import load_dataset
import librosa
import librosa.display
import IPython.display as ipd

# Configure plotting
plt.style.use('ggplot')
%matplotlib inline

## 1.1 Download Dataset
Downloading RAVDESS from HuggingFace.

In [ ]:
print('Downloading RAVDESS dataset...')
dataset = load_dataset('narad/ravdess')
print(dataset)

## 1.2 Data Preparation
Extracting metadata and audio paths.

In [ ]:
train_data = dataset['train']

# Convert to pandas dataframe for easier EDA
df = pd.DataFrame(train_data)
display(df.head())

Let's define emotion mapping based on RAVDESS identifier.

In [ ]:
emotion_mapping = {
    0: 'neutral',
    1: 'calm',
    2: 'happy',
    3: 'sad',
    4: 'angry',
    5: 'fearful',
    6: 'disgust',
    7: 'surprised'
}

df['emotion_label'] = df['labels'].map(emotion_mapping)
display(df.head())

## 1.3 Exploratory Data Analysis
### Emotion Distribution

In [ ]:
plt.figure(figsize=(10, 6))
sns.countplot(data=df, x='emotion_label', order=df['emotion_label'].value_counts().index)
plt.title('Distribution of Emotion Classes')
plt.xlabel('Emotion')
plt.ylabel('Count')
plt.xticks(rotation=45)
plt.show()

As noted in the plan, `neutral` class is imbalanced. It has half the samples of other classes.

### Gender Distribution

In [ ]:
plt.figure(figsize=(8, 5))
sns.countplot(data=df, x='speaker_gender')
plt.title('Distribution of Speaker Gender')
plt.xlabel('Gender')
plt.ylabel('Count')
plt.show()

### Audio Visualization

In [ ]:
def plot_waveform_and_spectrogram(audio_array, sr, title):
    plt.figure(figsize=(14, 5))
    
    plt.subplot(1, 2, 1)
    librosa.display.waveshow(audio_array, sr=sr)
    plt.title(f'Waveform: {title}')
    
    plt.subplot(1, 2, 2)
    D = librosa.amplitude_to_db(np.abs(librosa.stft(audio_array)), ref=np.max)
    librosa.display.specshow(D, y_axis='log', x_axis='time', sr=sr)
    plt.title(f'Spectrogram: {title}')
    plt.colorbar(format='%+2.0f dB')
    
    plt.tight_layout()
    plt.show()

In [ ]:
# Let's visualize one example for each emotion
unique_emotions = df['emotion_label'].unique()

for emotion in unique_emotions:
    sample = df[df['emotion_label'] == emotion].iloc[0]
    audio_array = np.array(sample['audio']['array'])
    sr = sample['audio']['sampling_rate']
    
    print(f'Emotion: {emotion.capitalize()}')
    display(ipd.Audio(audio_array, rate=sr))
    plot_waveform_and_spectrogram(audio_array, sr, emotion.capitalize())
    break # Only show one for now, remove break to see all